# 🚀 ComfyUI + MiniMax-H3 on Google Colab (A100 GPU Edition)

Google AI Pro 等のプランで付与される **Colab Compute Units (CU)** を活用して、強力な最新動画生成モデル **MiniMax-H3 (Hailuo)** を A100 (40GB VRAM) 環境で動かすためのオールインワン検証テンプレートです。

> 📖 **詳細な解説・検証記事 (Zenn)**: [Google AI Pro(2,900円)に課金するとA100 40GBがColab経由で毎月37時間分使える話](https://zenn.dev/grand2/articles/ddba80ba400f6f)

### 💡 特徴・ポイント
- **A100 GPU 最適化**: MiniMax-H3 の大規模モデル (`int8_convrot` 等) + Text Encoder + Audio/Video VAE をストレスなくロード。
- **🚀 Turbo LoRA 最適化 (推奨)**: 通常 25 ステップ（約30分）かかる生成を、**わずか 6〜8 ステップ（約7分・所要時間 1/4）へ激変** させる Turbo LoRA を自動配備。
- **高速化スタック対応**: `SageAttention`、`TeaCache` などの最新高速化ノード群をトグル1つで自動セットアップ。
- **Google Drive 完全永続化 (モデルキャッシュ & 出力動画)**:
  - モデル・LoRA（約20GB）を `MyDrive/ComfyUI_Models/` にキャッシュし、2回目以降のDLをスキップ（起動待ち0秒）。
  - 生成された動画・画像は **`MyDrive/ComfyUI_Outputs/` に自動保存**。ランタイムが切断・終了しても成果物が消えません。
- **Cloudflare Tunnel (無料・トークン不要)**: ngrok 不要ですぐにセキュアな一時公開 URL (`trycloudflare.com`) を自動発行。

> ⚠️ **注意**: ノートブック上部のメニュー「ランタイム」→「ランタイムのタイプを変更」から、**GPU (A100)** が選択されていることを確認してください。

## Step 1: 環境確認 & Google Drive マウント (永続化連携)

In [1]:
# GPU および CUDA バージョンの確認 (A100 がアサインされているか確認)
!nvidia-smi

# モデルキャッシュ & 出力動画を Google Drive に永続保存
USE_GOOGLE_DRIVE = True  # @param {type:"boolean"}

if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    print("✅ Google Drive mounted successfully!")

Fri Sep 25 15:19:32 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   33C    P0             51W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

## Step 2: ComfyUI 本体 & 高速化スタックのセットアップ

- **ENABLE_ACCELERATION (推奨)**: `SageAttention`、`TeaCache` などの最新高速化ノードを有効化。
- **CUDA 12.8 / 13.0 互換性ハンドリング**: PyTorch cu128 とホスト CUDA 13.0 の互換レイヤーを安全に処理します。

In [2]:
import os

# @title 高速化オプション設定
ENABLE_ACCELERATION = True  # @param {type:"boolean"}
INSTALL_TEACACHE = True     # @param {type:"boolean"}

%cd /content

# ComfyUI 本体のクローン (最新版)
if not os.path.exists("/content/ComfyUI"):
    !git clone https://github.com/comfyanonymous/ComfyUI.git
else:
    %cd /content/ComfyUI
    !git pull
    %cd /content

# 依存ライブラリのインストール
%cd /content/ComfyUI
!pip install -q -r requirements.txt
!pip install -q huggingface_hub

# SageAttention & Triton セットアップ
if ENABLE_ACCELERATION:
    print("⚡ 高速化スタック (SageAttention, Triton 等) をセットアップ中...")
    !pip install -q triton sageattention

# ComfyUI-Manager のインストール
%cd /content/ComfyUI/custom_nodes
if not os.path.exists("/content/ComfyUI/custom_nodes/ComfyUI-Manager"):
    !git clone https://github.com/ltdrdata/ComfyUI-Manager.git

# MiniMax-H3 Easy ノード
if not os.path.exists("/content/ComfyUI/custom_nodes/ComfyUI-MiniMaxH3-Easy"):
    !git clone https://github.com/kijai/ComfyUI-MiniMaxH3-Easy.git || true

# TeaCache 高速化ノード
if INSTALL_TEACACHE and not os.path.exists("/content/ComfyUI/custom_nodes/ComfyUI-MiniMaxH3-TeaCache"):
    !git clone https://github.com/chengzeyi/ComfyUI-MiniMaxH3-TeaCache.git || true

%cd /content/ComfyUI
print("✅ ノードおよび依存ライブラリの準備が完了しました！")

/content
Cloning into 'ComfyUI'...
remote: Enumerating objects: 52807, done.
remote: Counting objects: 100% (473/473), done.
remote: Compressing objects: 100% (172/172), done.
remote: Total 52807 (delta 391), reused 301 (delta 301), pack-reused 52334 (from 4)
Receiving objects: 100% (52807/52807), 91.38 MiB | 15.92 MiB/s, done.
Resolving deltas: 100% (36075/36075), done.
/content/ComfyUI
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.2/25.2 MB 92.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.5/19.5 MB 108.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.2/47.2 MB 47.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 432.7/432.7 kB 41.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.7/77.7 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 118.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.1/100.1 MB 25.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.2/94.2

## Step 3: MiniMax-H3 モデル & Turbo LoRA の準備 (Google Drive 連携)

- **Turbo LoRA (`DOWNLOAD_TURBO_LORA = True`)**: `lightx2v/Minimax-h3-Turbo` をダウンロードし、**8ステップサンプリング（生成時間1/4）** を可能にします。
- `USE_GOOGLE_DRIVE = True` の場合：
  - **モデル & LoRA キャッシュ**: `/content/drive/MyDrive/ComfyUI_Models/` (次回以降 DL スキップ)
  - **動画出力先**: `/content/drive/MyDrive/ComfyUI_Outputs/` (インスタンス終了後も成果物を保持)

In [3]:
import os
from huggingface_hub import hf_hub_download

DOWNLOAD_TURBO_LORA = True

if USE_GOOGLE_DRIVE:
    MODELS_BASE = "/content/drive/MyDrive/ComfyUI_Models"
    OUTPUT_DRIVE = "/content/drive/MyDrive/ComfyUI_Outputs"
    os.makedirs(MODELS_BASE, exist_ok=True)
    os.makedirs(OUTPUT_DRIVE, exist_ok=True)

    # 1. モデル群を Google Drive に連携
    for sub in ["diffusion_models", "text_encoders", "vae", "loras"]:
        drive_sub = os.path.join(MODELS_BASE, sub)
        local_sub = os.path.join("/content/ComfyUI/models", sub)
        os.makedirs(drive_sub, exist_ok=True)
        if os.path.exists(local_sub) and not os.path.islink(local_sub):
            !rm -rf {local_sub}
        if not os.path.exists(local_sub):
            os.symlink(drive_sub, local_sub)

    # 2. 【二刀流設計】表の扉(normal)はDriveへ、裏の扉(secret)は使い捨てへ
    base_output = "/content/ComfyUI/output"
    if os.path.islink(base_output):
        os.unlink(base_output)
    os.makedirs(base_output, exist_ok=True)

    normal_dir = os.path.join(base_output, "normal")
    secret_dir = os.path.join(base_output, "secret")

    # normal フォルダだけを Google Drive に直結
    if not os.path.exists(normal_dir):
        os.symlink(OUTPUT_DRIVE, normal_dir)
    # secret フォルダは Drive に繋がずローカル隔離
    os.makedirs(secret_dir, exist_ok=True)

    print("🚪 二刀流の保存ゲートを構築しました：")
    print("  - 通常モード (normal/)   👉 Google Drive に自動保存")
    print("  - シークレット (secret/) 👉 Google Drive を完全遮断（ローカル直保存）")

print("✅ セットアップ完了！")


🚪 二刀流の保存ゲートを構築しました：
  - 通常モード (normal/)   👉 Google Drive に自動保存
  - シークレット (secret/) 👉 Google Drive を完全遮断（ローカル直保存）
✅ セットアップ完了！


## Step 4: ComfyUI 起動 & Cloudflare Tunnel 経由でアクセス

バックグラウンドで ComfyUI を起動し、Cloudflare Tunnel (`trycloudflare.com`) の安全なパブリック URL を発行して**起動状態を維持**します。
表示された `https://xxxx.trycloudflare.com` のリンクをクリックすると ComfyUI の WebUI が開きます。

### 💡 Turbo LoRA を使った超高速ワークフローの組み方
1. ComfyUI を開いたら、モデルローダー（`UNETLoader`）の直後に **`LoraLoader`** を挟みます。
2. LoRA に **`minimax_h3_fl2v_turbo_8step_...safetensors`** を選択します。
3. **KSampler の steps を `25` → `8` に変更** します（Euler / normal 推奨）。
4. これで **約 6〜8 分で 1 本の美麗動画が完成** します！

In [ ]:
import subprocess
import threading
import time
import re

# Cloudflared のダウンロード
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /content/cloudflared
!chmod +x /content/cloudflared

# ComfyUI を安定したクラシック画面＋CORS許可で起動
print("⚡ Starting ComfyUI (Stable Mode)...")
%cd /content/ComfyUI
comfy_proc = subprocess.Popen([
    "python", "main.py",
    "--listen", "0.0.0.0",
    "--port", "8188",
    "--highvram",
    "--enable-cors-header", "*",
    "--front-end-version", "Comfy-Org/ComfyUI_legacy_frontend@latest"
], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

# ComfyUI の起動ログを別スレッドで監視
def log_comfy():
    for line in iter(comfy_proc.stdout.readline, ''):
        print("[ComfyUI]", line, end="")

threading.Thread(target=log_comfy, daemon=True).start()

# 起動完了を待ってからトンネルを起動
time.sleep(10)
print("🌐 Starting Cloudflare Tunnel...")
tunnel_proc = subprocess.Popen([
    "/content/cloudflared", "tunnel",
    "--url", "http://127.0.0.1:8188"
], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

url_found = False
for line in iter(tunnel_proc.stdout.readline, ''):
    match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
    if match and not url_found:
        url = match.group(0)
        print("\n" + "="*60)
        print(f"🎉 ComfyUI is LIVE: {url}")
        print("="*60 + "\n")
        print("💡 安定モードで起動しました。このURLを開いてください。")
        url_found = True

try:
    tunnel_proc.wait()
except KeyboardInterrupt:
    print("\n⏹️ サーバーを停止しました。")
    comfy_proc.terminate()
    tunnel_proc.terminate()


⚡ Starting ComfyUI (Stable Mode)...
/content/ComfyUI
[ComfyUI] [INFO] setup plugin alembic.autogenerate.schemas
[ComfyUI] [INFO] setup plugin alembic.autogenerate.tables
[ComfyUI] [INFO] setup plugin alembic.autogenerate.types
[ComfyUI] [INFO] setup plugin alembic.autogenerate.constraints
[ComfyUI] [INFO] setup plugin alembic.autogenerate.defaults
[ComfyUI] [INFO] setup plugin alembic.autogenerate.comments
[ComfyUI] [INFO] setup plugin alembic.ext.checkconstraint_byname
[ComfyUI] [START] Security scan
[ComfyUI] [DONE] Security scan
[ComfyUI] ## ComfyUI-Manager: installing dependencies.
🌐 Starting Cloudflare Tunnel...
[ComfyUI] ## ComfyUI-Manager: installing dependencies done.
[ComfyUI] ** ComfyUI startup time: 2026-09-25 15:22:46.089
[ComfyUI] ** Platform: Linux
[ComfyUI] ** Python version: 3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]
[ComfyUI] ** Python executable: /usr/bin/python3
[ComfyUI] ** ComfyUI Path: /content/ComfyUI
[ComfyUI] ** ComfyUI Base Folder Path: /content/ComfyU

In [5]:
from huggingface_hub import hf_hub_download
import os

print("📥 網目ノイズの出ない最高峰モデル (Scaled FP8) をダウンロード中...")
hf_hub_download(
    repo_id="Comfy-Org/MiniMax-H3",
    filename="diffusion_models/minimax_h3_fl2va_pruned_fp8_scaled.safetensors",
    local_dir="/content/drive/MyDrive/ComfyUI_Models",
    local_dir_use_symlinks=False
)
print("✅ ダウンロード完了！Google Driveに永久保存されました。")


📥 網目ノイズの出ない最高峰モデル (Scaled FP8) をダウンロード中...


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


diffusion_models/minimax_h3_fl2va_pruned(…): reconstructing file:   0%|          |  0.00B / 21.0GB            

diffusion_models/minimax_h3_fl2va_pruned(…): downloading bytes:           |  0.00B            

✅ ダウンロード完了！Google Driveに永久保存されました。


## 🛑 (作業完了時) 確実に課金をストップする方法

動画生成の検証が完了したら、**最も確実に Compute Units (CU) の消費をストップ** するため、以下の手順でインスタンスを破棄してください：

### 💡 【推奨・確実度 100%】Colab メニューから切断
1. Colab 画面上部メニューの **「ランタイム」** をクリック
2. **「ランタイムの接続を解除して削除」** を選択（確認画面で「はい」）

> 成果物（動画）は Google Drive（`MyDrive/ComfyUI_Outputs/`）にリアルタイム保存されているため、切断しても安全です。